In [2]:
import pandas as pd
import numpy as np
import pprint
import random

from sklearn.model_selection import train_test_split  
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

######################## MODELS ########################

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

########################################################

from sklearn.model_selection import RandomizedSearchCV

RANDOM_STATE = 777
TEST_SIZE = 0.2

In [3]:
df = pd.read_csv("spambase_csv.csv")

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4601 entries, 0 to 4600
Data columns (total 58 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   word_freq_make              4601 non-null   float64
 1   word_freq_address           4601 non-null   float64
 2   word_freq_all               4601 non-null   float64
 3   word_freq_3d                4601 non-null   float64
 4   word_freq_our               4601 non-null   float64
 5   word_freq_over              4601 non-null   float64
 6   word_freq_remove            4601 non-null   float64
 7   word_freq_internet          4601 non-null   float64
 8   word_freq_order             4601 non-null   float64
 9   word_freq_mail              4601 non-null   float64
 10  word_freq_receive           4601 non-null   float64
 11  word_freq_will              4601 non-null   float64
 12  word_freq_people            4601 non-null   float64
 13  word_freq_report            4601 non-null   

In [5]:
NUMBER_OF_FEATURES = len(df.columns)

In [6]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

### Logistic Regression Pipeline Implementation

In [7]:
base_logistic_regression_classifier = LogisticRegression(
    C=0.7,
    solver="lbfgs",
    max_iter=5000,
    l1_ratio=0,
)

In [8]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# NOTE: There is no categorical column in our dataset,
# NOTE: so I am skipping this phase.
# cat_pipeline = Pipeline([
#     ("imputer", SimpleImputer(strategy="most_frequent")),
#     ("encoder", OneHotEncoder(handle_unknown="ignore"))
# ])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_features),
    # ("cat", cat_pipeline, categorical_features)
])

logistic_regression_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", base_logistic_regression_classifier)
])

In [9]:
logistic_regression_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

In [10]:
def print_basic_metrics(model_name: str, y_test, y_pred) -> None:
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)

    print("="*50)
    print(f"{model_name.upper()} PERFORMANCE METRICS")
    print("="*50)
    print(f"Accuracy:                {accuracy:.3f}")
    print(f"Precision (weighted):    {precision:.3f}")
    print(f"Recall (weighted):       {recall:.3f}")
    print(f"F1-Score (weighted):     {f1:.3f}")
    print("="*50)

    print("\nConfusion Matrix:")
    print("-----------------")
    print(f"True Negatives:  {cm[0,0]}")
    print(f"False Positives: {cm[0,1]}")
    print(f"False Negatives: {cm[1,0]}")
    print(f"True Positives:  {cm[1,1]}")
    print("="*50)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

In [11]:
y_pred = logistic_regression_pipeline.predict(X_test)
print_basic_metrics(model_name="logistic regression", y_test=y_test, y_pred=y_pred)

LOGISTIC REGRESSION PERFORMANCE METRICS
Accuracy:                0.914
Precision (weighted):    0.925
Recall (weighted):       0.914
F1-Score (weighted):     0.914

Confusion Matrix:
-----------------
True Negatives:  508
False Positives: 27
False Negatives: 52
True Positives:  334

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       535
           1       0.93      0.87      0.89       386

    accuracy                           0.91       921
   macro avg       0.92      0.91      0.91       921
weighted avg       0.91      0.91      0.91       921



#### Randomized Search of Best Hyperparamters

In [12]:
param_grid = {
    "classifier__C": np.linspace(start=0.001, stop=1, num=30),
    "classifier__solver": ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
    "classifier__max_iter": [num for num in range(100, 5000, 250)]
}

random_logistic_regression = RandomizedSearchCV(
    logistic_regression_pipeline,
    param_distributions=param_grid,
    n_iter=10,                       # NOTE: Number of random combinations
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_logistic_regression.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=5000))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__C': array([0.001 ..., 1. ]), 'classifier__max_iter': [100, 350, ...], 'classifier__solver': ['lbfgs', 'newton-cg', ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used h

In [13]:
print("Best Parameters:", random_logistic_regression.best_params_)
print("Best CV Score:", random_logistic_regression.best_score_)

best_logistic_regression = random_logistic_regression.best_estimator_
y_pred = best_logistic_regression.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))

Best Parameters: {'classifier__solver': 'newton-cholesky', 'classifier__max_iter': 4350, 'classifier__C': np.float64(0.8966551724137931)}
Best CV Score: 0.9228260869565217
Test Accuracy: 0.9131378935939196


In [14]:
print_basic_metrics(model_name="tuned logistic regression", y_test=y_test, y_pred=y_pred)

TUNED LOGISTIC REGRESSION PERFORMANCE METRICS
Accuracy:                0.913
Precision (weighted):    0.920
Recall (weighted):       0.913
F1-Score (weighted):     0.913

Confusion Matrix:
-----------------
True Negatives:  506
False Positives: 29
False Negatives: 51
True Positives:  335

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       535
           1       0.92      0.87      0.89       386

    accuracy                           0.91       921
   macro avg       0.91      0.91      0.91       921
weighted avg       0.91      0.91      0.91       921



#### Making Logistic Regression Less Complex

In [15]:
classifier = best_logistic_regression.named_steps["classifier"]

In [16]:
importances = np.abs(classifier.coef_[0])
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

word_freq_george              3.783025
word_freq_hp                  2.334188
word_freq_cs                  1.618136
char_freq_%24                 1.397661
word_freq_meeting             1.388354
word_freq_hpl                 1.181199
word_freq_85                  1.101690
word_freq_edu                 1.055998
word_freq_project             1.019115
word_freq_lab                 1.011156
word_freq_remove              0.957975
capital_run_length_longest    0.936001
word_freq_re                  0.889254
word_freq_conference          0.808751
word_freq_000                 0.806284
word_freq_free                0.802729
char_freq_%23                 0.763779
word_freq_3d                  0.747370
word_freq_credit              0.453785
word_freq_data                0.441557
word_freq_business            0.440138
capital_run_length_total      0.391154
word_freq_our                 0.389841
capital_run_length_average    0.373833
word_freq_technology          0.338398
char_freq_%3B            

In [17]:
important_features_scores = feature_importance[feature_importance > 0.5]
important_features = feature_importance[feature_importance > 0.5].index.to_list()
print(important_features_scores)

word_freq_george              3.783025
word_freq_hp                  2.334188
word_freq_cs                  1.618136
char_freq_%24                 1.397661
word_freq_meeting             1.388354
word_freq_hpl                 1.181199
word_freq_85                  1.101690
word_freq_edu                 1.055998
word_freq_project             1.019115
word_freq_lab                 1.011156
word_freq_remove              0.957975
capital_run_length_longest    0.936001
word_freq_re                  0.889254
word_freq_conference          0.808751
word_freq_000                 0.806284
word_freq_free                0.802729
char_freq_%23                 0.763779
word_freq_3d                  0.747370
dtype: float64


In [18]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

In [19]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# NOTE: There is no categorical column in our dataset,
# NOTE: so I am skipping this phase.
# cat_pipeline = Pipeline([
#     ("imputer", SimpleImputer(strategy="most_frequent")),
#     ("encoder", OneHotEncoder(handle_unknown="ignore"))
# ])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_features),
    # ("cat", cat_pipeline, categorical_features)
])

logistic_regression_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", base_logistic_regression_classifier)
])

In [20]:
logistic_regression_pipeline.fit(X_train_optimized, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

In [21]:
y_pred = logistic_regression_pipeline.predict(X_test)
print_basic_metrics(model_name="logistic regression", y_test=y_test, y_pred=y_pred)

LOGISTIC REGRESSION PERFORMANCE METRICS
Accuracy:                0.882
Precision (weighted):    0.909
Recall (weighted):       0.882
F1-Score (weighted):     0.880

Confusion Matrix:
-----------------
True Negatives:  504
False Positives: 31
False Negatives: 78
True Positives:  308

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.94      0.90       535
           1       0.91      0.80      0.85       386

    accuracy                           0.88       921
   macro avg       0.89      0.87      0.88       921
weighted avg       0.88      0.88      0.88       921



In [22]:
print(NUMBER_OF_FEATURES)
print(len(important_features))

58
18


#### NOTES for Logistic Regression

The feature importance ranking reveals a compelling narrative about spam detection. George (3.78) dominates - likely a reference to George W. Bush (but I am not sure), whose frequent mention in early 2000s spam campaigns made it a strong discriminator. HP-related terms (hp, hpl) and "cs" follow closely, showing the technology marketing origins of many spam emails.

What's interesting is the prominence of structural features like the dollar sign (1.40), "#" (0.76), and longest capital run (0.94). These patterns capture the exaggerated formatting and financial appeals typical of spam - "FREE" and "remove" also rank highly, mirroring the classic "unsubscribe" tactic.

Before optimization, the model used 58 features and achieved ~0.91 accuracy. After pruning to 18 features - retaining only those with strongest signals - accuracy settled at 0.89. The 0.02 decline is negligible (is almost nothing), but the simplification is substantial: fewer features mean faster inference, lower memory footprint, and reduced overfitting risk. This represents a classic bias-variance trade-off where a marginal performance cost delivers significant practical gains in classifier efficiency and interpretability.

### KNN

In [23]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

In [24]:
knn_base_classifier = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="minkowski",
    p=2
)

In [25]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_features),
    ("cat", cat_pipeline, categorical_features)
])

knn_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", KNeighborsClassifier())
])

knn_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transform

In [26]:
y_pred = knn_pipeline.predict(X_test)
print_basic_metrics(model_name="K-Nearest Neighbors", y_test=y_test, y_pred=y_pred)

K-NEAREST NEIGHBORS PERFORMANCE METRICS
Accuracy:                0.894
Precision (weighted):    0.916
Recall (weighted):       0.894
F1-Score (weighted):     0.893

Confusion Matrix:
-----------------
True Negatives:  506
False Positives: 29
False Negatives: 69
True Positives:  317

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91       535
           1       0.92      0.82      0.87       386

    accuracy                           0.89       921
   macro avg       0.90      0.88      0.89       921
weighted avg       0.90      0.89      0.89       921



#### Randomized Search of Best Hyperparamters

In [27]:
param_grid = {
    "classifier__n_neighbors": list(range(3, 25)),
    "classifier__weights": ["uniform", "distance"],
    "classifier__p": [1, 2],
    "classifier__algorithm": ['auto', 'ball_tree', 'kd_tree', 'brute']
}

random_knn = RandomizedSearchCV(
    knn_pipeline,
    param_distributions=param_grid,
    n_iter=5,                   # NOTE: Number of random combinations
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42
)

random_knn.fit(X_train, y_train)

best_knn= random_knn.best_estimator_
y_pred_random = best_knn.predict(X_test)

print("Best Parameters (Random):", random_knn.best_params_)

Best Parameters (Random): {'classifier__weights': 'distance', 'classifier__p': 2, 'classifier__n_neighbors': 16, 'classifier__algorithm': 'auto'}


In [28]:
print_basic_metrics(model_name="Tuned K-Nearest Neighbors", y_test=y_test, y_pred=y_pred)

TUNED K-NEAREST NEIGHBORS PERFORMANCE METRICS
Accuracy:                0.894
Precision (weighted):    0.916
Recall (weighted):       0.894
F1-Score (weighted):     0.893

Confusion Matrix:
-----------------
True Negatives:  506
False Positives: 29
False Negatives: 69
True Positives:  317

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91       535
           1       0.92      0.82      0.87       386

    accuracy                           0.89       921
   macro avg       0.90      0.88      0.89       921
weighted avg       0.90      0.89      0.89       921



### Random Forest Classifier

In [29]:
classifier_metrics = dict()

#### Random Forest Pipeline

In [30]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

In [31]:
rf = RandomForestClassifier(
	n_estimators=500,
	max_depth=20, 
	min_samples_split=5,
	min_samples_leaf=2, 
	max_features="sqrt",
	bootstrap=True, 
	n_jobs=-1, 
	random_state=RANDOM_STATE
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
random_forest_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("classifier", rf)
])

random_forest_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transform

In [32]:
y_pred = random_forest_pipeline.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name="Random Forest Classifier", y_test=y_test, y_pred=y_pred)

RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.933
Precision (weighted):    0.943
Recall (weighted):       0.933
F1-Score (weighted):     0.932

Confusion Matrix:
-----------------
True Negatives:  514
False Positives: 21
False Negatives: 41
True Positives:  345

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94       535
           1       0.94      0.89      0.92       386

    accuracy                           0.93       921
   macro avg       0.93      0.93      0.93       921
weighted avg       0.93      0.93      0.93       921



In [33]:
classifier_metrics["Random Forest Base Pipeline"] = {
    "accuracy": rf_accuracy,
    "no_features": 58
}

In [34]:
pprint.pprint(classifier_metrics)

{'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58}}


#### Random Search of Hyperparameters for Random Forest

In [35]:
param_dist = {  
	"classifier__n_estimators": [100, 200, 500],  
	"classifier__max_depth": [None, 10, 20, 30],  
	"classifier__min_samples_split": [2, 5, 10],  
	"classifier__min_samples_leaf": [1, 2, 4],  
	"classifier__max_features": ["sqrt", "log2"]  
}  
  
random_random_forest = RandomizedSearchCV(  
	random_forest_pipeline,  
	param_distributions=param_dist,  
	n_iter=20,  
	cv=5,  
	scoring="accuracy",  
	n_jobs=-1,  
	random_state=42  
)  
  
random_random_forest.fit(X_train, y_train)  

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._state=777))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__max_depth': [None, 10, ...], 'classifier__max_features': ['sqrt', 'log2'], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the var

In [36]:
best_random_forest = random_random_forest.best_estimator_  
classifer = best_random_forest.named_steps["classifier"]

print("Best Parameters:", random_random_forest.best_params_)
print("Best CV Score:", random_random_forest.best_score_)

y_pred = best_random_forest.predict(X_test)
tuned_rf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name="Tuned Random Forest Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'classifier__n_estimators': 200, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 1, 'classifier__max_features': 'log2', 'classifier__max_depth': 30}
Best CV Score: 0.9538043478260869
TUNED RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.946
Precision (weighted):    0.952
Recall (weighted):       0.946
F1-Score (weighted):     0.946

Confusion Matrix:
-----------------
True Negatives:  517
False Positives: 18
False Negatives: 32
True Positives:  354

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95       535
           1       0.95      0.92      0.93       386

    accuracy                           0.95       921
   macro avg       0.95      0.94      0.94       921
weighted avg       0.95      0.95      0.95       921



In [37]:
classifier_metrics["Tuned Random Forest"] = {
    "accuracy": tuned_rf_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58}}


#### Optimizing Random Forest

In [38]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

char_freq_%21                 0.107670
capital_run_length_average    0.075169
char_freq_%24                 0.071830
word_freq_free                0.064465
word_freq_remove              0.060242
capital_run_length_longest    0.054236
word_freq_your                0.051724
capital_run_length_total      0.050939
word_freq_hp                  0.042302
word_freq_money               0.038980
word_freq_you                 0.034430
word_freq_our                 0.032908
word_freq_000                 0.026559
word_freq_hpl                 0.020101
word_freq_george              0.017617
word_freq_edu                 0.015870
word_freq_1999                0.014950
word_freq_receive             0.013988
word_freq_internet            0.013340
char_freq_%28                 0.013049
word_freq_over                0.012236
word_freq_will                0.011734
word_freq_all                 0.011664
word_freq_mail                0.011597
word_freq_email               0.010547
word_freq_re             

In [39]:
important_features_scores = feature_importance[feature_importance > 0.05]
important_features = feature_importance[feature_importance > 0.05].index.to_list()
print(important_features_scores)

char_freq_%21                 0.107670
capital_run_length_average    0.075169
char_freq_%24                 0.071830
word_freq_free                0.064465
word_freq_remove              0.060242
capital_run_length_longest    0.054236
word_freq_your                0.051724
capital_run_length_total      0.050939
dtype: float64


In [40]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

In [41]:
rf = RandomForestClassifier(
	n_estimators=500,
	max_depth=20, 
	min_samples_split=5,
	min_samples_leaf=2, 
	max_features="sqrt",
	bootstrap=True, 
	n_jobs=-1, 
	random_state=RANDOM_STATE
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
random_forest_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("classifier", rf)
])

random_forest_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transform

In [42]:
y_pred = random_forest_pipeline.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name="Tuned and Optimized Random Forest Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.908
Precision (weighted):    0.934
Recall (weighted):       0.908
F1-Score (weighted):     0.907

Confusion Matrix:
-----------------
True Negatives:  512
False Positives: 23
False Negatives: 62
True Positives:  324

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.96      0.92       535
           1       0.93      0.84      0.88       386

    accuracy                           0.91       921
   macro avg       0.91      0.90      0.90       921
weighted avg       0.91      0.91      0.91       921



In [43]:
classifier_metrics["Tuned and Optimized Random Forest Classifier"] = {
    "accuracy": rf_accuracy,
    "no_features": len(important_features_scores)
}

In [44]:
classifier_metrics

{'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
  'no_features': 8}}

#### Random Forest vs. Other classifiers

#### Saving classifier

### Extreme Random Forest

In [45]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Extreme Random Forest Pipeline

In [46]:
from sklearn.ensemble import ExtraTreesClassifier

In [47]:
xrf = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=False,
    criterion="gini",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

In [48]:
num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
extreme_random_forest_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("xrf_classifier", xrf)
])

extreme_random_forest_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('xrf_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [50]:
y_pred = extreme_random_forest_pipeline.predict(X_test)
xrf_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics["Extreme Random Forest Classifier"] = {
    "accuracy": xrf_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name="Extreme Random Forest Classifier", y_test=y_test, y_pred=y_pred)

EXTREME RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.955
Precision (weighted):    0.960
Recall (weighted):       0.955
F1-Score (weighted):     0.955

Confusion Matrix:
-----------------
True Negatives:  520
False Positives: 15
False Negatives: 26
True Positives:  360

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       535
           1       0.96      0.93      0.95       386

    accuracy                           0.96       921
   macro avg       0.96      0.95      0.95       921
weighted avg       0.96      0.96      0.96       921



In [51]:
pprint.pprint(classifier_metrics)

{'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
                                                  'no_features': 8}}


#### Random Search of Hyperparameters for Extreme Random Forest

In [56]:
# extended hyperparameter tuning space
param_dist = {
    "xrf_classifier__n_estimators": [100, 200, 300, 500, 800],
    "xrf_classifier__criterion": ["gini", "entropy", "log_loss"],
    "xrf_classifier__max_depth": [None, 10, 20, 30, 50],
    "xrf_classifier__min_samples_split": [2, 5, 10, 20],
    "xrf_classifier__min_samples_leaf": [1, 2, 4, 8],
    "xrf_classifier__max_features": ["sqrt", "log2", None, 0.5, 0.8],
    "xrf_classifier__bootstrap": [True],
    "xrf_classifier__max_samples": [None, 0.5, 0.7, 0.9],
    "xrf_classifier__min_impurity_decrease": [0.0, 0.01, 0.05],
    "xrf_classifier__ccp_alpha": [0.0, 0.001, 0.01],
    "xrf_classifier__class_weight": [None, "balanced", "balanced_subsample"]
}

search = RandomizedSearchCV(
    extreme_random_forest_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._state=777))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'xrf_classifier__bootstrap': [True], 'xrf_classifier__ccp_alpha': [0.0, 0.001, ...], 'xrf_classifier__class_weight': [None, 'balanced', ...], 'xrf_classifier__criterion': ['gini', 'entropy', ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User G

In [58]:
best_extreme_random_forest = search.best_estimator_  
classifer = best_extreme_random_forest.named_steps["xrf_classifier"]

print("Best Parameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

y_pred = best_extreme_random_forest.predict(X_test)
tuned_xrf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name="Tuned Random Forest Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'xrf_classifier__n_estimators': 200, 'xrf_classifier__min_samples_split': 2, 'xrf_classifier__min_samples_leaf': 2, 'xrf_classifier__min_impurity_decrease': 0.0, 'xrf_classifier__max_samples': 0.5, 'xrf_classifier__max_features': 0.5, 'xrf_classifier__max_depth': 20, 'xrf_classifier__criterion': 'log_loss', 'xrf_classifier__class_weight': None, 'xrf_classifier__ccp_alpha': 0.0, 'xrf_classifier__bootstrap': True}
Best CV Score: 0.9426630434782609
TUNED RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.934
Precision (weighted):    0.940
Recall (weighted):       0.934
F1-Score (weighted):     0.934

Confusion Matrix:
-----------------
True Negatives:  513
False Positives: 22
False Negatives: 39
True Positives:  347

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94       535
           1       0.94      0.90      0.92       386

    accuracy                           0.93       

In [59]:
classifier_metrics["Tuned Extreme Random Forest"] = {
    "accuracy": tuned_xrf_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
                                 'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
                                                  'no_features': 8}}


#### Optimizing Extreme Random Forest

In [60]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

char_freq_%24                 0.078482
word_freq_remove              0.078191
word_freq_your                0.072690
char_freq_%21                 0.070127
word_freq_hp                  0.062518
capital_run_length_longest    0.053313
word_freq_free                0.048878
word_freq_000                 0.047398
capital_run_length_average    0.036053
word_freq_george              0.033069
word_freq_our                 0.032912
word_freq_money               0.029924
word_freq_you                 0.027134
word_freq_hpl                 0.026575
capital_run_length_total      0.024649
word_freq_edu                 0.017042
word_freq_receive             0.016872
word_freq_business            0.016543
word_freq_1999                0.016066
word_freq_over                0.015618
word_freq_internet            0.015264
word_freq_all                 0.014374
word_freq_re                  0.013286
word_freq_will                0.012943
word_freq_order               0.011452
word_freq_email          

In [74]:
important_features_scores = feature_importance[feature_importance > 0.035]
important_features = feature_importance[feature_importance > 0.035].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

char_freq_%24                 0.078482
word_freq_remove              0.078191
word_freq_your                0.072690
char_freq_%21                 0.070127
word_freq_hp                  0.062518
capital_run_length_longest    0.053313
word_freq_free                0.048878
word_freq_000                 0.047398
capital_run_length_average    0.036053
dtype: float64
NUMBER OF FEATURES: 9


In [75]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

In [76]:
num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
extreme_random_forest_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("xrf_classifier", xrf)
])

extreme_random_forest_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('xrf_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [77]:
y_pred = extreme_random_forest_pipeline.predict(X_test)
xrf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name="Tuned and Optimized Extreme Random Forest Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED EXTREME RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.932
Precision (weighted):    0.935
Recall (weighted):       0.932
F1-Score (weighted):     0.931

Confusion Matrix:
-----------------
True Negatives:  511
False Positives: 24
False Negatives: 39
True Positives:  347

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94       535
           1       0.94      0.90      0.92       386

    accuracy                           0.93       921
   macro avg       0.93      0.93      0.93       921
weighted avg       0.93      0.93      0.93       921



In [78]:
classifier_metrics["Tuned and Optimized Extreme Random Forest Classifier"] = {
    "accuracy": xrf_accuracy,
    "no_features": len(important_features_scores)
}

In [79]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Random Forest', {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Extreme Random Forest',
  {'accuracy': 0.9337676438653637, 'no_features': 58}),
 ('Random Forest Base Pipeline',
  {'accuracy': 0.9326818675352877, 'no_features': 58}),
 ('Tuned and Optimized Extreme Random Forest Classifier',
  {'accuracy': 0.9315960912052117, 'no_features': 9}),
 ('Tuned and Optimized Random Forest Classifier',
  {'accuracy': 0.9077090119435396, 'no_features': 8})]


### Multilayer Perceptron

In [81]:
from sklearn.neural_network import MLPClassifier  

In [82]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Multilayer Perceptron Pipeline

In [84]:
mlp = MLPClassifier(  
	hidden_layer_sizes=(64, 32),  
	activation="relu",  
	solver="adam",  
	max_iter=300,  
	random_state=RANDOM_STATE  
) 

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
mlp_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("mlp_classifier", xrf)
])

mlp_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('mlp_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [85]:
y_pred = mlp_pipeline.predict(X_test)
mlp_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics["Multilayer Perceptor Classifier"] = {
    "accuracy": mlp_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name="Multilayer Perceptor Classifier", y_test=y_test, y_pred=y_pred)

MULTILAYER PERCEPTOR CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.955
Precision (weighted):    0.960
Recall (weighted):       0.955
F1-Score (weighted):     0.955

Confusion Matrix:
-----------------
True Negatives:  520
False Positives: 15
False Negatives: 26
True Positives:  360

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       535
           1       0.96      0.93      0.95       386

    accuracy                           0.96       921
   macro avg       0.96      0.95      0.95       921
weighted avg       0.96      0.96      0.96       921



In [86]:
pprint.pprint(classifier_metrics)

{'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9554831704668838,
                                     'no_features': 58},
 'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
                                 'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'accuracy': 0.9315960912052117,
                                                          'no_features': 9},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
                                                  'no_features': 8}}


#### Random Search of Hyperparameters for Multiplayer Perceptor

#### Optimizing Multilayer Perceptor